# Task 1

### Importing libraries

In [ ]:
import torch
import torch.nn as nn 
import open_clip
from torch.utils.data import Subset, DataLoader, TensorDataset
from torchvision import datasets, models, transforms
from sklearn.model_selection import train_test_split

torch.manual_seed(6304)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(6304)

### Loading models

In [ ]:
resnet_weights = models.ResNet50_Weights.IMAGENET1K_V2 #loading resnet model
resnet_preprocess = resnet_weights.transforms()
resnet_50 = models.resnet50(weights=resnet_weights)

vit_weights = models.ViT_B_16_Weights.IMAGENET1K_V1 #loading ViT
vit_preprocess = vit_weights.transforms()
vit_model = models.vit_b_16(weights='ViT_B_16_Weights.IMAGENET1K_V1')

clip_model, training_preprocess_clip, validation_preprocess_clip = open_clip.create_model_and_transforms(
    "ViT-B-32",
    pretrained='openai'
) #loading CLIP

for parameter in resnet_50.parameters(): # freezing the models
    parameter.requires_grad = False

for parameter in vit_model.parameters():
    parameter.requires_grad = False

for parameter in clip_model.parameters():
    parameter.requires_grad = False



### Loading datasets + dataloader for ResNet-50 training

In [ ]:
stl_train_resnet = datasets.STL10(root='./data', split='train', download=True, transform=resnet_preprocess) #load datasets

index_train = []
labels_train = stl_train_resnet.labels

for i in range(len(stl_train_resnet)):
    index_train.append(i)

train_index, val_index = train_test_split(index_train, test_size=0.2, random_state=6304, stratify=labels_train)
train_set = Subset(stl_train_resnet, train_index) #splitting based on index
val_set = Subset(stl_train_resnet, val_index)

train_dataloader_resnet = DataLoader(train_set, batch_size=64, shuffle=True, transforms=resnet_preprocess)
val_dataloader_resnet = DataLoader(val_set, batch_size=64, shuffle=False, transforms=resnet_preprocess)


### Loading datasets + dataloader for ViT training

In [ ]:
stl_train_vit = datasets.STL10(root='./data', split='train', download=True, transform=vit_preprocess) #load datasets

index_train = []
labels_train = stl_train_vit.labels

for i in range(len(stl_train_vit)):
    index_train.append(i)

train_index, val_index = train_test_split(index_train, test_size=0.2, random_state=6304, stratify=labels_train)
train_set = Subset(stl_train_vit, train_index)
val_set = Subset(stl_train_vit, val_index)

train_dataloader_vit = DataLoader(train_set, batch_size=64, shuffle=True, transforms=resnet_preprocess)
val_dataloader_vit = DataLoader(val_set, batch_size=64, shuffle=False, transforms=resnet_preprocess)


### Loading datasets + dataloader for CLIP training

In [ ]:
stl_train_clip = datasets.STL10(root='./data', split='train', download=True, transform=training_preprocess_clip) #load datasets
stl_val_clip = datasets.STL10(root='./data', split='train', download=True, transform=validation_preprocess_clip) 

index_train = []
labels_train = stl_train_clip.labels

for i in range(len(stl_train_vit)):
    index_train.append(i)

train_index, val_index = train_test_split(index_train, test_size=0.2, random_state=6304, stratify=labels_train)
train_set = Subset(stl_train_clip, train_index)
val_set = Subset(stl_val_clip, val_index)

train_dataloader_clip = DataLoader(train_set, batch_size=64, shuffle=True)
val_dataloader_clip = DataLoader(val_set, batch_size=64, shuffle=False)


### Setup classification layers for training

In [ ]:
resnet_layer = nn.Linear(resnet_50.fc.in_features, 10)
vit_layer = nn.Linear(vit_model.heads.head.in_features, 10)
clip_layer = nn.Linear(clip_model.visual.output_dim, 10)


### Collect representations + turn representations into dataloader

In [ ]:
resnet_50.fc = nn.Identity()
vit_model.heads.head = nn.Identity()

resnet_train_latent = []
renset_val_latent = []

vit_train_latent = []
vit_val_latent = []

clip_train_latent = []
clip_val_latent = []

train_labels = []
val_labels = []

with torch.no_grad(): # inference and collecting latent (z) into lists
    resnet_50.eval()
    for images, labels in train_dataloader_resnet:
        z = resnet_50(images)
        resnet_train_latent.append(z)
        train_labels.append(labels)

    for images, labels in val_dataloader_resnet:
        z = resnet_50(images)
        renset_val_latent.append(z)
        val_labels.append(labels)

    vit_model.eval()
    for images, labels in train_dataloader_vit:
        z = vit_model(images)
        vit_train_latent.append(z)

    for images, labels in val_dataloader_vit:
        z = vit_model(images)
        vit_val_latent.append(z)

    clip_model.eval()
    for images, labels in train_dataloader_clip:
        z = clip_model.encode_image(images, normalize=True)
        clip_train_latent.append(z)

    for images, labels in val_dataloader_clip:
        z = clip_model.encode_image(images, normalize=True)
        clip_val_latent.append(z)

#converting latent into a dataloader... for classification training
z_train_resnet = torch.cat(resnet_train_latent, dim=0)
z_val_resnet = torch.cat(renset_val_latent, dim=0)

z_train_vit = torch.cat(vit_train_latent, dim=0)
z_val_vit = torch.cat(vit_val_latent, dim=0)

z_train_clip = torch.cat(clip_train_latent, dim=0)
z_val_clip = torch.cat(clip_val_latent, dim=0)

train_labels_torch = torch.cat(train_labels, dim=0)
val_labels_torch = torch.cat(val_labels, dim=0)


resnet_train_dataset = TensorDataset(z_train_resnet, train_labels_torch)
resnet_val_dataset = TensorDataset(z_val_resnet, val_labels_torch)

vit_train_dataset = TensorDataset(z_train_vit, train_labels_torch)
vit_val_dataset = TensorDataset(z_val_vit, val_labels_torch)

clip_train_dataset = TensorDataset(z_train_clip, train_labels_torch)
clip_val_dataset = TensorDataset(z_val_clip, val_labels_torch)


resnet_z_dataloader_train = DataLoader(resnet_train_dataset, batch_size=64, shuffle=True)
renset_z_dataloader_val = DataLoader(resnet_val_dataset, batch_size=64, shuffle=False)

vit_z_dataloader_train = DataLoader(vit_train_dataset, batch_size=64, shuffle=True)
vit_z_dataloader_val = DataLoader(vit_val_dataset, batch_size=64, shuffle=False)

clip_z_dataloader_train = DataLoader(clip_train_dataset, batch_size=64, shuffle=True)
clip_z_dataloader_val = DataLoader(clip_val_dataset, batch_size=64, shuffle=False)

### Hyperparamaters for training classification layer

In [ ]:
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4

MAX_EPOCHS = 50
EARLY_STOPPING_PATIENCE = 5

SEED = 6304

### Train classification layer for ResNet-50